# P3. 지출 예측

목적: 고객별 **이번 달 말 예상 지출**, **다음 달 대분류별 지출 예측**, **백테스트 및 MAE 비교**를 수행한다.

설계 기준:
- 이번 달 말 예상: 현재까지 지출 ÷ 경과일수 × 해당 월 일수
- 다음 달 예측 기본 모델: 최근 3개월 이동평균
- 비교 모델: 최근 3개월 선형 추세 (월 전체 지출 기준, 6~12장)
- `src/forecast.py` 모델: 대분류별 SMA vs WMA(가중 1:2:3) 비교 (13장)
- 1~11월 데이터를 이용해 다음 달을 예측하고 실제값과 비교
- MAE가 낮은 모델을 확인

`src/forecast.py`의 함수와 동일한 데이터를 사용해 Pandas 결과를 검증한다.

In [1]:
# 1. 라이브러리 및 프로젝트 경로 설정
from pathlib import Path
import sys
import calendar

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# VS Code에서 notebooks 폴더를 기준으로 실행하는 경우 프로젝트 루트로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /home/claude/repo


In [2]:
# 2. 데이터 불러오기
# 원본 CSV를 직접 읽지 않고 P1 공통 로더를 사용한다 (중복 943건 제거, 취소 6,403건 제외, 카테고리 정규화).
from loader import load_clean

df = load_clean(verbose=True)
print("컬럼:", df.columns.tolist())
display(df.head())

원본 318,145행 → 중복 943 제거 → 취소 6,403 제외 → 310,799행
컬럼: ['transaction_id', 'customer_id', 'persona_id', 'age_group', 'gender', 'occupation', 'annual_income', 'household_type', 'transaction_datetime', 'merchant_name', 'category', 'amount', 'payment_method', 'card_product', 'transaction_status', 'dt', 'ym', 'tx_date', 'income_band', 'category_std', 'big_category', 'merchant_filled']


,transaction_id,customer_id,persona_id,age_group,gender,occupation,annual_income,household_type,transaction_datetime,merchant_name,...,payment_method,card_product,transaction_status,dt,ym,tx_date,income_band,category_std,big_category,merchant_filled
0,TX000556,C0001,P01,30s,M,OFFICE_WORKER,37410000,SINGLE,2025-01-01 19:41:24,배달의민족,...,CREDIT_CARD,CARD_A,APPROVED,2025-01-01 19:41:24,2025-01,2025-01-01,4천만 미만,DELIVERY,배달,배달의민족
1,TX001253,C0001,P01,30s,M,OFFICE_WORKER,37410000,SINGLE,2025-01-02 19:24:09,동네식당,...,CREDIT_CARD,CARD_A,APPROVED,2025-01-02 19:24:09,2025-01,2025-01-02,4천만 미만,DINING,식비,동네식당
2,TX001553,C0001,P01,30s,M,OFFICE_WORKER,37410000,SINGLE,2025-01-03 08:24:59,동네로스터리,...,CREDIT_CARD,CARD_A,APPROVED,2025-01-03 08:24:59,2025-01,2025-01-03,4천만 미만,CAFE,카페,동네로스터리
3,TX004245,C0001,P01,30s,M,OFFICE_WORKER,37410000,SINGLE,2025-01-06 10:59:25,아파트관리비,...,CREDIT_CARD,CARD_A,APPROVED,2025-01-06 10:59:25,2025-01,2025-01-06,4천만 미만,UTILITY,주거통신,아파트관리비
4,TX004259,C0001,P01,30s,M,OFFICE_WORKER,37410000,SINGLE,2025-01-06 11:11:07,세븐일레븐,...,CHECK_CARD,CARD_A,APPROVED,2025-01-06 11:11:07,2025-01,2025-01-06,4천만 미만,CONVENIENCE,식비,세븐일레븐


In [3]:
# 3. P3에서 사용할 공통 컬럼 정리
# P1 loader가 이미 dt/ym을 만들어 주는 경우 그대로 사용한다.
# 아직 loader를 사용하지 않는 경우를 대비해 필요한 컬럼을 생성한다.

work = df.copy()

if "dt" not in work.columns:
    if "transaction_datetime" not in work.columns:
        raise ValueError("dt 또는 transaction_datetime 컬럼이 필요합니다.")
    work["dt"] = pd.to_datetime(work["transaction_datetime"], errors="coerce")
else:
    work["dt"] = pd.to_datetime(work["dt"], errors="coerce")

work = work.dropna(subset=["dt"])

if "ym" not in work.columns:
    work["ym"] = work["dt"].dt.strftime("%Y-%m")

work["amount"] = pd.to_numeric(work["amount"], errors="coerce").fillna(0)

if "big_category" not in work.columns:
    if "category" not in work.columns:
        raise ValueError("big_category 또는 category 컬럼이 필요합니다.")
    work["big_category"] = work["category"].astype(str).str.strip().str.upper()

print(work[["customer_id", "dt", "ym", "amount", "big_category"]].head())

  customer_id                  dt       ym  amount big_category
0       C0001 2025-01-01 19:41:24  2025-01    7760           배달
1       C0001 2025-01-02 19:24:09  2025-01   20400           식비
2       C0001 2025-01-03 08:24:59  2025-01    6960           카페
3       C0001 2025-01-06 10:59:25  2025-01  228300         주거통신
4       C0001 2025-01-06 11:11:07  2025-01    2150           식비


## 4. 데이터 기본 확인

P3 예측을 실행하기 전에 날짜 범위, 고객 수, 월별 데이터가 정상인지 확인한다.

In [4]:
print("데이터 기간:", work["dt"].min(), "~", work["dt"].max())
print("고객 수:", work["customer_id"].nunique())
print("월 수:", work["ym"].nunique())

monthly_check = (
    work.groupby("ym", as_index=False)["amount"]
    .sum()
    .sort_values("ym")
)

display(monthly_check)

데이터 기간: 2025-01-01 00:00:37 ~ 2025-12-31 23:59:36
고객 수: 1000
월 수: 12


,ym,amount
0,2025-01,1620591360
1,2025-02,1418357680
2,2025-03,1579590520
3,2025-04,1536957640
4,2025-05,1587449920
5,2025-06,1411443840
6,2025-07,1628726940
7,2025-08,1677810820
8,2025-09,1636622790
9,2025-10,1759508420


## 5. 월별·대분류별 지출 집계

다음 달 예측의 기본 단위는 **고객 × 월 × 대분류**다.

In [5]:
monthly_category = (
    work.groupby(
        ["customer_id", "ym", "big_category"],
        as_index=False
    )["amount"]
    .sum()
    .rename(columns={"amount": "monthly_amount"})
    .sort_values(["customer_id", "ym", "big_category"])
)

display(monthly_category.head(20))

,customer_id,ym,big_category,monthly_amount
0,C0001,2025-01,배달,144460
1,C0001,2025-01,쇼핑,118000
2,C0001,2025-01,식비,298960
3,C0001,2025-01,여가,57480
4,C0001,2025-01,여행,14500
5,C0001,2025-01,의료,12200
6,C0001,2025-01,주거통신,350400
7,C0001,2025-01,카페,12900
8,C0001,2025-02,교통,18300
9,C0001,2025-02,배달,210600


## 6. 최근 3개월 이동평균 함수

예측 대상 월의 직전 3개월을 사용한다. 데이터가 부족한 초기 구간은 백테스트에서 제외한다.

In [6]:
def moving_average_predict(history_values, window=3):
    values = pd.Series(history_values).dropna().tail(window)
    if len(values) < window:
        return np.nan
    return float(values.mean())


def linear_trend_predict(history_values, window=3):
    values = pd.Series(history_values).dropna().tail(window).to_numpy(dtype=float)
    if len(values) < window:
        return np.nan

    x = np.arange(len(values))
    slope, intercept = np.polyfit(x, values, 1)
    prediction = slope * len(values) + intercept
    return max(0.0, float(prediction))

## 7. 대분류별 다음 달 예측

특정 고객의 기준 월을 넣으면 다음 달의 대분류별 이동평균/선형추세 예측값을 만든다.

In [7]:
def forecast_next_month_pandas(customer_df, base_month):
    base_date = pd.to_datetime(base_month + "-01")
    next_month = (base_date + pd.DateOffset(months=1)).strftime("%Y-%m")

    monthly = (
        customer_df.groupby(["ym", "big_category"], as_index=False)["amount"]
        .sum()
        .rename(columns={"amount": "monthly_amount"})
    )

    results = []

    for category, group in monthly.groupby("big_category"):
        group = group.sort_values("ym")
        history = group[group["ym"] < next_month].tail(3)

        ma = moving_average_predict(history["monthly_amount"], 3)
        trend = linear_trend_predict(history["monthly_amount"], 3)

        results.append({
            "big_category": category,
            "moving_average": ma,
            "linear_trend": trend,
        })

    result = pd.DataFrame(results)
    if not result.empty:
        result["selected_forecast"] = result["moving_average"]

    return next_month, result

In [8]:
# 8. 실제 고객 1명으로 다음 달 예측 확인

CUSTOMER_ID = "C0003"   # README 리포트 예시 고객
BASE_MONTH = "2025-12"

customer_df = work[
    work["customer_id"].astype(str) == CUSTOMER_ID
].copy()

next_month, forecast_table = forecast_next_month_pandas(
    customer_df,
    BASE_MONTH
)

print("고객 ID:", CUSTOMER_ID)
print("기준 월:", BASE_MONTH)
print("예측 월:", next_month)

display(forecast_table)

if not forecast_table.empty:
    print("이동평균 전체 예측:", forecast_table["moving_average"].sum())
    print("선형추세 전체 예측:", forecast_table["linear_trend"].sum())

고객 ID: C0003
기준 월: 2025-12
예측 월: 2026-01


,big_category,moving_average,linear_trend,selected_forecast
0,교육,201473.333333,203293.333333,201473.333333
1,교통,65426.666667,24056.666667,65426.666667
2,구독,48690.000000,48690.000000,48690.000000
3,기타,36423.333333,35493.333333,36423.333333
4,배달,29600.000000,44700.000000,29600.000000
5,쇼핑,300600.000000,334000.000000,300600.000000
6,식비,315610.000000,842750.000000,315610.000000
7,여가,17336.666667,0.000000,17336.666667
8,여행,NaN,NaN,NaN
9,의료,24193.333333,0.000000,24193.333333


이동평균 전체 예측: 1091326.6666666667
선형추세 전체 예측: 1577366.6666666667


## 9. 이번 달 말 예상 지출

공식:

`현재까지 지출 ÷ 경과 일수 × 해당 월 일수`

고정비는 P5 결과를 연결한 뒤 별도로 더한다. 현재 노트북에서는 변동지출 페이스만 계산한다.

In [9]:
def current_month_forecast_pandas(customer_df, month):
    month_df = customer_df[customer_df["ym"] == month].copy()

    month_start = pd.to_datetime(month + "-01")
    days_in_month = calendar.monthrange(month_start.year, month_start.month)[1]

    if month_df.empty:
        return {
            "actual_to_date": 0.0,
            "elapsed_days": 0,
            "days_in_month": days_in_month,
            "forecast": 0.0,
        }

    actual_to_date = float(month_df["amount"].sum())
    elapsed_days = int(month_df["dt"].max().day)

    forecast = (
        actual_to_date / elapsed_days * days_in_month
        if elapsed_days > 0 else 0.0
    )

    return {
        "actual_to_date": actual_to_date,
        "elapsed_days": elapsed_days,
        "days_in_month": days_in_month,
        "forecast": float(forecast),
    }


# 전체 데이터로 계산하면 12월 31일까지 다 들어 있어 예상값 = 실제값이 된다.
# 팀 기준일(2025-12-20)까지 자른 데이터로 '이번 달 진행 중' 상황을 만든다.
AS_OF = "2025-12-20"
customer_df_asof = customer_df[customer_df["dt"] <= pd.Timestamp(AS_OF) + pd.Timedelta(days=1)]
customer_df_asof = customer_df_asof[customer_df_asof["dt"].dt.strftime("%Y-%m-%d") <= AS_OF]

current_month_result = current_month_forecast_pandas(
    customer_df_asof,
    BASE_MONTH
)

display(pd.DataFrame([current_month_result]))

,actual_to_date,elapsed_days,days_in_month,forecast
0,576610.0,20,31,893745.5


## 10. 백테스트

1~11월 데이터를 사용해 다음 달을 예측하고 실제값과 비교한다.

- 1~3월 → 4월
- 2~4월 → 5월
- ...
- 9~11월 → 12월

대분류별 예측값을 합산해 고객의 월 전체 지출을 예측한다.

In [10]:
def backtest_customer(customer_df, min_history=3):
    monthly = (
        customer_df.groupby("ym", as_index=False)["amount"]
        .sum()
        .rename(columns={"amount": "actual_amount"})
        .sort_values("ym")
    )

    months = monthly["ym"].tolist()
    results = []

    for i in range(min_history, len(months)):
        target_month = months[i]
        history = monthly.iloc[i-min_history:i]["actual_amount"]

        ma_pred = moving_average_predict(history, min_history)
        trend_pred = linear_trend_predict(history, min_history)
        actual = float(monthly.iloc[i]["actual_amount"])

        results.append({
            "actual_month": target_month,
            "actual_amount": actual,
            "moving_average": ma_pred,
            "linear_trend": trend_pred,
            "ma_error": abs(ma_pred - actual),
            "trend_error": abs(trend_pred - actual),
        })

    return pd.DataFrame(results)


backtest = backtest_customer(customer_df)
display(backtest)

,actual_month,actual_amount,moving_average,linear_trend,ma_error,trend_error
0,2025-04,446050.0,834320.000000,1.327670e+06,388270.000000,8.816200e+05
1,2025-05,705300.0,769780.000000,4.855600e+05,64480.000000,2.197400e+05
2,2025-06,413000.0,761456.666667,3.337367e+05,348456.666667,7.926333e+04
3,2025-07,1389320.0,521450.000000,4.884000e+05,867870.000000,9.009200e+05
4,2025-08,966360.0,835873.333333,1.519893e+06,130486.666667,5.535333e+05
5,2025-09,380620.0,922893.333333,1.476253e+06,542273.333333,1.095633e+06
6,2025-10,1093970.0,912100.000000,0.000000e+00,181870.000000,1.093970e+06
7,2025-11,1100260.0,813650.000000,9.412600e+05,286610.000000,1.590000e+05
8,2025-12,807000.0,858283.333333,1.577923e+06,51283.333333,7.709233e+05


In [11]:
# 11. MAE 비교

if backtest.empty:
    print("백테스트에 필요한 최소 4개월 데이터가 없습니다.")
else:
    mae_ma = backtest["ma_error"].mean()
    mae_trend = backtest["trend_error"].mean()

    mae_result = pd.DataFrame({
        "model": ["3개월 이동평균", "3개월 선형추세"],
        "MAE": [mae_ma, mae_trend],
    })

    display(mae_result)

    print(f"이동평균 MAE: {mae_ma:,.0f}원")
    print(f"선형추세 MAE: {mae_trend:,.0f}원")
    print("※ 실제 프로젝트에서는 이 결과를 기준으로 최종 모델을 결정한다.")

,model,MAE
0,3개월 이동평균,317955.555556
1,3개월 선형추세,639400.370370


이동평균 MAE: 317,956원
선형추세 MAE: 639,400원
※ 실제 프로젝트에서는 이 결과를 기준으로 최종 모델을 결정한다.


## 12. 모든 고객 기준 백테스트

특정 고객 한 명이 아니라 전체 고객을 대상으로 모델 성능을 확인한다.

In [12]:
all_backtests = []

for customer_id, group in work.groupby("customer_id"):
    result = backtest_customer(group)

    if result.empty:
        continue

    result["customer_id"] = customer_id
    all_backtests.append(result)

if all_backtests:
    all_backtest_df = pd.concat(all_backtests, ignore_index=True)

    overall_mae = pd.DataFrame({
        "model": ["3개월 이동평균", "3개월 선형추세"],
        "MAE": [
            all_backtest_df["ma_error"].mean(),
            all_backtest_df["trend_error"].mean(),
        ],
    })

    display(overall_mae)
else:
    print("백테스트 가능한 고객이 없습니다.")

,model,MAE
0,3개월 이동평균,7.356868e+05
1,3개월 선형추세,1.106095e+06


## 13. `src/forecast.py` 결과 확인

최종 모듈의 공통 인터페이스인 `analyze(df, customer_id, month)`가 정상적으로 동작하는지 확인한다.

In [13]:
import forecast

full_result = forecast.analyze(df, CUSTOMER_ID, BASE_MONTH)
mid_result = forecast.analyze(load_clean(as_of=AS_OF), CUSTOMER_ID, BASE_MONTH)

print("[전체 데이터]  ", full_result["message"])
print(f"[{AS_OF} 기준]", mid_result["message"])
print("고객 백테스트:", mid_result["data"]["backtest"])
display(pd.DataFrame({"SMA": mid_result["data"]["sma_forecast_by_cat"],
                      "WMA": mid_result["data"]["wma_forecast_by_cat"]}))

[전체 데이터]   12월에 총 80만 7천 원 썼어요. 지난달보다 29만 3천 원 적어요. 다음 달에는 식비에 31만 5천 원 정도 쓸 것으로 보여요.
[2025-12-20 기준] 이 속도면 12월에 87만 2천 원을 쓰게 돼요. 지난달보다 22만 7천 원 적어요. 다음 달에는 쇼핑에 30만 원 정도 쓸 것으로 보여요.
고객 백테스트: {'mae_sma': 76678, 'mae_wma': 80622, 'selected_model': 'SMA (단순 이동평균)', 'improvement': '기본 모델 채택'}


,SMA,WMA
교육,190867,276300
교통,69577,69230
구독,48690,48690
기타,2673,1337
배달,29770,25902
쇼핑,300600,306167
식비,150157,188435
여가,14797,10013
의료,993,497
주거통신,29700,29700


## 13-1. 대분류별 SMA vs WMA 백테스트 (전체 고객, 4~12월)

`forecast.backtest_all()`로 고객 × 대분류 × 월마다 직전 3개월 SMA·WMA 예측값과 실제값을 비교한다. 한 번이라도 지출이 있었던 고객·대분류 조합만 쓰고, 지출이 없는 달은 0으로 채운다.

In [14]:
months = [f"2025-{m:02d}" for m in range(4, 13)]
bt = forecast.backtest_all(df, months)

overall = pd.DataFrame({"model": ["SMA (단순 이동평균)", "WMA (가중 이동평균 1:2:3)"],
                        "MAE": [bt["err_sma"].mean(), bt["err_wma"].mean()]}).round(0)
display(overall)
by_month = bt.groupby("ym")[["err_sma", "err_wma"]].mean().round(0)
display(by_month)
print(f"WMA가 SMA보다 나은 달: {(by_month['err_wma'] < by_month['err_sma']).sum()} / {len(by_month)}개월")
print("→ 최종 모델:", "WMA" if bt['err_wma'].mean() < bt['err_sma'].mean() else "SMA (forecast.FINAL_MODEL)")

,model,MAE
0,SMA (단순 이동평균),114815.0
1,WMA (가중 이동평균 1:2:3),116475.0


,err_sma,err_wma
ym,,
2025-04,111008.0,112385.0
2025-05,113924.0,116527.0
2025-06,103554.0,105415.0
2025-07,109445.0,109498.0
2025-08,112028.0,112062.0
2025-09,112946.0,117340.0
2025-10,119865.0,121292.0
2025-11,126592.0,127627.0
2025-12,123976.0,126132.0


WMA가 SMA보다 나은 달: 0 / 9개월
→ 최종 모델: SMA (forecast.FINAL_MODEL)


## 14. SQL ↔ Pandas 교차검증

`sql/queries_P3.sql`을 SQLite에서 실행해 같은 수치가 나오는지 확인한다. SQL은 원본 CSV에서 정제·정규화를 독립적으로 다시 수행한다.

In [15]:
import re, sqlite3

con = sqlite3.connect(":memory:")
pd.read_csv(PROJECT_ROOT / "data" / "transactions.csv").to_sql("transactions", con, index=False)
sql = (PROJECT_ROOT / "sql" / "queries_P3.sql").read_text(encoding="utf-8")
blocks = {p.split("]", 1)[0]: p.split("]", 1)[1] for p in re.split(r"^-- \[", sql, flags=re.M)[1:]}
con.executescript(blocks.pop("SETUP"))
run = lambda key, **kw: pd.read_sql(blocks[next(k for k in blocks if k.startswith(key))], con, params=kw or None)

checks = []
# V1 다음 달 대분류별 예측 (고객 1명)
q2 = run("P3-02", customer_id=CUSTOMER_ID).set_index("big_category")
py = forecast.predict_next_month_categories(df[df["customer_id"] == CUSTOMER_ID], "2026-01")
m = q2.join(pd.DataFrame({"sma": py["sma_series"], "wma": py["wma_series"]}), how="outer").fillna(0)
d = max((m["sma_forecast"] - m["sma"].round()).abs().max(), (m["wma_forecast"] - m["wma"].round()).abs().max())
checks.append(("V1 다음 달 예측 SMA·WMA", d == 0, f"{len(m)}개 대분류, 최대 차이 {d}"))
# V2 전체 백테스트 MAE
q3 = run("P3-03")
ok = int(q3["n"][0]) == len(bt) and round(bt["err_sma"].mean()) == q3["mae_sma"][0] and round(bt["err_wma"].mean()) == q3["mae_wma"][0]
checks.append(("V2 백테스트 MAE", ok, f"SQL n={int(q3['n'][0]):,}, SMA {q3['mae_sma'][0]:,.0f} / WMA {q3['mae_wma'][0]:,.0f}"))
# V3 이번 달 말 예상
q5 = run("P3-05", customer_id=CUSTOMER_ID, month=BASE_MONTH, as_of=AS_OF)
ok = int(q5["projected_total"][0]) == mid_result["data"]["projected_total"]
checks.append(("V3 이번 달 말 예상", ok, f"SQL {int(q5['projected_total'][0]):,}원 / Python {mid_result['data']['projected_total']:,}원"))

pd.DataFrame(checks, columns=["검증", "통과", "상세"])

,검증,통과,상세
0,V1 다음 달 예측 SMA·WMA,True,"12개 대분류, 최대 차이 0.0"
1,V2 백테스트 MAE,True,"SQL n=101,556, SMA 114,815 / WMA 116,475"
2,V3 이번 달 말 예상,True,"SQL 872,411원 / Python 872,411원"


## 14. 검증 체크리스트

- [ ] `transactions.csv` 정상 로드
- [ ] `dt`, `ym`, `amount`, `big_category` 확인
- [ ] 고객 × 월 × 대분류 집계 확인
- [ ] 이번 달 말 예상 지출 계산 확인
- [ ] 최근 3개월 이동평균 계산 확인
- [ ] 선형 추세 계산 확인
- [ ] 1~11월 백테스트 수행
- [ ] MAE 비교
- [ ] `src/forecast.py`의 `analyze()` 실행 확인
- [ ] SQL 결과와 Pandas 핵심 수치 교차검증

최종 발표에서는 실제 고객 1명의 예측 결과와 백테스트 MAE를 근거로 설명한다.